# Executed direct multi-horizon forecasts

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import root_mean_squared_error

root = Path.cwd() if Path("README.md").exists() else Path.cwd().parent
m = json.loads((root / "reports/metrics.json").read_text())
[(h, v["selected_on_validation"]) for h, v in m["horizons"].items()]

[('1', 'random_forest'), ('7', 'ridge')]

In [2]:
p = pd.read_csv(root / "reports/test-predictions.csv", parse_dates=["origin", "target_date"])
for h, group in p.groupby("horizon_days"):
    result = m["horizons"][str(h)]
    rmse = root_mean_squared_error(group.target, group.selected_prediction)
    assert abs(rmse - result["test"][result["selected_on_validation"]]["rmse"]) < 1e-10
    assert ((group.target_date - group.origin).dt.days == h).all()
p.groupby("horizon_days").size()

horizon_days
1    544
7    532
dtype: int64

In [3]:
coverage = pd.read_csv(root / "reports/coverage.csv")
coverage

,year,calendar_days,observed_soil10_days,observed_fraction
0,2016,366,365,0.997268
1,2017,365,364,0.997260
2,2018,365,347,0.950685
3,2019,365,354,0.969863
4,2020,366,366,1.000000
5,2021,365,363,0.994521
6,2022,365,325,0.890411
7,2023,365,358,0.980822
8,2024,366,188,0.513661


In [4]:
b = pd.read_csv(
    root / "reports/backtest.csv", parse_dates=["train_target_max", "first_validation_origin"]
)
assert (b.train_target_max < b.first_validation_origin).all()
b[["horizon_days", "validation_year", "method", "rmse"]]

,horizon_days,validation_year,method,rmse
0,1,2021,persistence,0.023392
1,1,2021,seasonal_climatology,0.091652
2,1,2021,ridge,0.020074
3,1,2021,random_forest,0.021442
4,1,2022,persistence,0.037845
5,1,2022,seasonal_climatology,0.099080
6,1,2022,ridge,0.033186
7,1,2022,random_forest,0.031353
8,7,2021,persistence,0.067010
9,7,2021,seasonal_climatology,0.093103


Only actual origin/target measurements enter metrics. No future weather or target imputation. Residual intervals have measured coverage and width, without an exchangeability guarantee.